# Session 2: thermal resistance, layer by layer
**A. Real-world problem.** A laptop chip dissipates 20 W. Its heat crosses a smear of thermal
paste, then an aluminium heatsink base, and is finally carried off by air blowing over the fins.
Is the chip safe? If its temperature must stay below 85 C, what do you change: the metal, its
thickness, or the airflow?

**Plan.** (1) turn each layer into a thermal resistance, (2) add resistances in series,
(3) find the chip temperature, (4) ask which resistance matters most, (5) solve a design problem.
Heat-transfer model only: general education, not a certified thermal design.

In [ ]:
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"
T_AMBIENT = 25.0        # deg C, air entering the fan
A_BASE = 4.0e-4         # m^2: 20 mm x 20 mm heatsink base (chip heat assumed spread evenly over it)
A_FINS = 1.0e-2         # m^2: total fin surface seen by the air (100 cm^2), a modelling assumption
PASTE_MM, K_PASTE = 0.2, 3.0   # thermal paste layer: thickness (mm) and conductivity W/(m K)

## Step 1: a resistance for every layer
Fourier's law Q = k A dT / L can be turned around into  dT = Q * R  (like Ohm's law, with
temperature difference in place of voltage and heat flow Q in place of current):

    conduction through a layer:   R = L / (k A)       (K/W)
    convection to moving air:     R = 1 / (h A)       (K/W)

Here h is the convection coefficient in W/(m^2 K): how well air carries heat off a surface
(natural convection about 5-15, a laptop fan about 25-100 in this simple model).
Resistances in a single heat path add, just like resistors in series.

In [ ]:
def r_cond(thickness_mm, k, area):
    """Thermal resistance (K/W) of a flat layer."""
    return thickness_mm * 1e-3 / (k * area)


def r_conv(h, area):
    """Thermal resistance (K/W) of a surface cooled by air with coefficient h."""
    return 1.0 / (h * area)


def stack(base_mm, k_base, h):
    """The three resistances of the chip-to-air path (K/W): paste, base, convection."""
    return [r_cond(PASTE_MM, K_PASTE, A_BASE), r_cond(base_mm, k_base, A_BASE), r_conv(h, A_FINS)]

## Step 2: chip temperature and the temperature at each interface
Total resistance R_total = R_paste + R_base + R_conv. The same Q = P (the chip power) crosses
every layer, so the chip (junction) temperature is  T_chip = T_ambient + P * R_total.
Each layer drops P * R_layer on the way.

In [ ]:
def chip_temperature(power, base_mm, k_base, h, t_amb=T_AMBIENT):
    return t_amb + power * sum(stack(base_mm, k_base, h))


def interface_temperatures(power, base_mm, k_base, h, t_amb=T_AMBIENT):
    """Temperatures (deg C) at chip, after paste, after base, then ambient air."""
    out, t = [], chip_temperature(power, base_mm, k_base, h, t_amb)
    out.append(t)
    for r in stack(base_mm, k_base, h)[:2]:
        t -= power * r
        out.append(t)
    out.append(t_amb)
    return out        # the drop across the convection film is the last gap, fins to air

## Step 3: the design problem
Given power P and a chip limit T_max, how much airflow (h) is needed? Rearranging
T_max = T_amb + P (R_paste + R_base + 1/(h A_fins)) gives the largest allowed convection
resistance and so the smallest h. The same idea gives the largest power for a given cooler.

In [ ]:
def required_h(power, t_max, base_mm, k_base, t_amb=T_AMBIENT):
    """Smallest h (W/(m^2 K)) that keeps the chip at or below t_max, or None if impossible."""
    r_left = (t_max - t_amb) / power - sum(stack(base_mm, k_base, 1.0)[:2])
    if r_left <= 0:
        return None
    return 1.0 / (r_left * A_FINS)


def max_power(t_max, base_mm, k_base, h, t_amb=T_AMBIENT):
    return (t_max - t_amb) / sum(stack(base_mm, k_base, h))

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    P, L, K, H = 20.0, 3.0, 200.0, 50.0
    rs = stack(L, K, H)
    print("resistances (K/W): paste %.4f, base %.4f, convection %.4f, total %.4f" % (*rs, sum(rs)))
    print("worked example: chip temperature = %.2f C" % chip_temperature(P, L, K, H))
    print("interface temperatures:", [round(t, 2) for t in interface_temperatures(P, L, K, H)])
    print("share of the total resistance:", [f"{100 * r / sum(rs):.1f}%" for r in rs])
    for name, k in (("steel", 50.0), ("aluminium", 200.0), ("copper", 400.0)):
        print(f"  base material {name:9s} k = {k:5.0f}: chip at {chip_temperature(P, L, k, H):.2f} C")
    for h in (25.0, 50.0, 100.0):
        print(f"  airflow h = {h:5.1f}: chip at {chip_temperature(P, L, K, h):.2f} C")
    for t in (1.0, 3.0, 6.0):
        print(f"  base thickness {t:3.1f} mm: chip at {chip_temperature(P, t, K, H):.2f} C")
    print("design: 30 W chip, limit 85 C needs h >= %.1f W/(m^2 K)" % required_h(30.0, 85.0, L, K))
    print("max power with this cooler at 85 C: %.1f W" % max_power(85.0, L, K, H))

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar(["paste", "base", "convection"], rs, color=["#e9c46a", "#8d99ae", "#2a9d8f"])
    ax.set_ylabel("thermal resistance (K/W)")
    ax.set_title("The air film dominates the path")
    ax.grid(alpha=0.3, axis="y")
    fig.tight_layout()
    fig.savefig(OUT / "resistances.png", dpi=120)

    h = np.linspace(20, 100, 81)
    fig2, ax2 = plt.subplots(figsize=(6, 4))
    for p in (10.0, 20.0, 30.0):
        ax2.plot(h, [chip_temperature(p, L, K, x) for x in h], label=f"{p:.0f} W")
    ax2.axhline(85.0, color="red", ls="--", label="85 C limit")
    ax2.set_xlabel("convection coefficient h (W/(m^2 K))")
    ax2.set_ylabel("chip temperature (C)")
    ax2.set_title("More airflow, cooler chip")
    ax2.grid(alpha=0.3)
    ax2.legend()
    fig2.tight_layout()
    fig2.savefig(OUT / "chip_vs_airflow.png", dpi=120)
    print("saved figures in", OUT)

## Where this stops
We treated heat as flowing through the whole base area evenly and the fins as one number.
Real chips are small hot spots on a larger base, so heat also spreads sideways. That needs 2D
conduction and time: Session 3.

## Coding tasks
1. **Easy.** Print the chip temperature for steel, aluminium and copper bases (k = 50, 200, 400) at h = 50 and compare the changes with the change from doubling h.
2. **Stretch.** Add a thermal pad layer to `stack` (1 mm, k = 5, over 4 cm^2) and print how many degrees it adds for a 20 W chip.
3. **Explain.** Explain, using R = 1/(hA), why doubling the fin area has the same effect on R_air as doubling h.

## Assessment
- **A. Conceptual.** The thermal paste is only 0.2 mm thick and the base is 3 mm thick, yet the air film has far more resistance than either. Why?
- **B. Numerical.** A 15 W chip sits on the same paste and a 4 mm copper base (k = 400), with h = 40 W/(m^2 K) on 100 cm^2 of fins. Find the chip temperature.
- **C. Graph and simulation interpretation.** On the airflow graph for 10, 20 and 30 W, which of those chips stay below 85 C at h = 50 W/(m^2 K)? Roughly where does the 30 W curve cross 85 C?
- **D. Design and optimisation challenge.** Design: a 30 W chip must stay at or below 85 C with the 3 mm aluminium base and h = 50. (a) Find the smallest h that works. (b) Would switching to a copper base (k = 400) be enough instead? (c) Suggest a design and a trade-off.

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    checks.append(("hand calculation: paste 0.2 mm, k = 3, 4 cm^2 has R = 0.1667 K/W",
                   abs(L.r_cond(0.2, 3.0, 4.0e-4) - 0.16667) < 1e-4))
    checks.append(("hand calculation: air film h = 50 on 100 cm^2 has R = 2 K/W",
                   abs(L.r_conv(50.0, 1.0e-2) - 2.0) < 1e-12))
    t = L.chip_temperature(20.0, 3.0, 200.0, 50.0)
    checks.append(("worked example: 20 W, 3 mm aluminium, h = 50 gives 69.08 C", abs(t - 69.08) < 0.01))
    temps = L.interface_temperatures(20.0, 3.0, 200.0, 50.0)
    checks.append(("interface temperatures fall step by step: 69.08, 65.75, 65.00, 25.00",
                   np.allclose(temps, [69.08, 65.75, 65.0, 25.0], atol=0.01)))
    checks.append(("the temperature drops add up to P * R_total",
                   abs((temps[0] - temps[-1]) - 20.0 * sum(L.stack(3.0, 200.0, 50.0))) < 1e-9))
    checks.append(("zero power leaves the chip at ambient", L.chip_temperature(0.0, 3.0, 200.0, 50.0) == L.T_AMBIENT))
    h = L.required_h(30.0, 85.0, 3.0, 200.0)
    checks.append(("design: a 30 W chip at 85 C needs h of about 55.7", abs(h - 55.68) < 0.05))
    checks.append(("with that h the chip sits exactly at the limit",
                   abs(L.chip_temperature(30.0, 3.0, 200.0, h) - 85.0) < 1e-9))
    checks.append(("an impossible limit returns None (200 W, 30 C)", L.required_h(200.0, 30.0, 3.0, 200.0) is None))
    checks.append(("max power of the example cooler at 85 C is 27.2 W", abs(L.max_power(85.0, 3.0, 200.0, 50.0) - 27.22) < 0.01))
    checks.append(("airflow matters more than metal: doubling h beats switching steel to copper",
                   L.chip_temperature(20.0, 3.0, 200.0, 25.0) - L.chip_temperature(20.0, 3.0, 200.0, 50.0)
                   > L.chip_temperature(20.0, 3.0, 50.0, 50.0) - L.chip_temperature(20.0, 3.0, 400.0, 50.0)))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
